# 00 - data audit, `gosplan`

The state of the transcription pipeline, and the coverage of the few series that arrived
already machine-readable.

**There is no inference in this notebook and there must never be any.** Nothing here fits a
model, tests a hypothesis or computes a detection statistic. In particular **no digit test
appears below and none may be added**: `docs/known_traps.md` trap 9 is that a digit test on a
badly transcribed table detects the transcription, and the gate on running one is a measured
double-transcription disagreement rate, which is itself a data-quality measurement and is the
only rate computed here.

Its job is to say which templates exist, which are filled, what the validator says about each,
how far two independent readings of the same page diverge, and which years and series the
machine-readable sources actually cover.

If `data/raw` is empty, every cell that reads acquired data prints what to run and does
nothing else. The transcription cells report the same way about
`data/transcription/`, which this repository ships empty because a template is written by a
person at run time. The catalogue cells (the target queue, the schema) describe code rather
than data and run either way.

In [ ]:
from __future__ import annotations

import zipfile
from pathlib import Path

import gosplan
import pandas as pd
from forensics_core.provenance import load_sources
from gosplan.transcribe import TARGETS, compare_files, read_filled, validate_file
from gosplan.transcribe.schema import (
    FIELD_NAMES,
    REQUIRED_FIELDS,
    TABLE_FIELDS,
    Confidence,
    CurrencyBasis,
    TerritorialBasis,
)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)

PROJECT = Path(gosplan.__file__).resolve().parents[2]
DATA = PROJECT / "data"
RAW = DATA / "raw"
TEMPLATES = DATA / "transcription" / "templates"
FILLED = DATA / "transcription" / "filled"

RUN_FIRST = (
    "data/raw is empty. From projects/gosplan run:\n"
    "    make data     # acquire the sources listed in data/SOURCES.yaml\n"
    "and then re-run this notebook. Nothing is computed until then."
)
TEMPLATES_FIRST = (
    "data/transcription/templates is empty. From projects/gosplan run:\n"
    "    make templates    # blank forms, one per target in gosplan.transcribe.targets\n"
    "Nothing about the transcription pipeline can be reported until then."
)
FILLED_FIRST = (
    "data/transcription/filled is empty. A template has to be filled in by a person before\n"
    "there is anything to validate or to compare. Transcribing target 1 twice, independently,\n"
    "is what produces the disagreement rate this project needs first."
)

RAW_FILES = (
    sorted(p for p in RAW.rglob("*") if p.is_file() and p.name != ".gitkeep")
    if RAW.is_dir()
    else []
)
RAW_EMPTY = not RAW_FILES
if RAW_EMPTY:
    print(RUN_FIRST)

## What is on disk, and what the registry records

The registry holds 57 entries and 23 of them have an acquirer. Every entry's
`verification.verdict` is `not_verified`: no independent second agent re-checked any of the 57
claims, so they are one careful session's findings and the first acquisition run is also the
check on them. The blocked list is long and worth reading rather than summarising: the CIA
Reading Room refuses scripted access outright, the Russian national digital library refuses
non-Russian egress, and the Russian State Archive of the Economy publishes its inventories but
hands over scans only on physical media in its reading room.

In [ ]:
if RAW_EMPTY:
    print(RUN_FIRST)
else:
    inventory = pd.DataFrame(
        [
            {
                "directory": p.parent.relative_to(RAW).as_posix() or ".",
                "suffix": p.suffix.lower(),
                "bytes": p.stat().st_size,
            }
            for p in RAW_FILES
        ]
    )
    display(
        inventory.groupby("directory")
        .agg(n_files=("bytes", "size"), total_bytes=("bytes", "sum"))
        .sort_index()
    )
    display(
        inventory.groupby("suffix")
        .agg(n_files=("bytes", "size"), total_bytes=("bytes", "sum"))
        .sort_values("n_files", ascending=False)
    )
    print(f"{len(RAW_FILES):,} files, {int(inventory['bytes'].sum()):,} bytes under data/raw")

In [ ]:
SOURCES = load_sources(DATA / "SOURCES.yaml")
registry = pd.DataFrame(
    [
        {
            "id": s.id,
            "access": s.access,
            "status": s.status,
            "verdict": (s.verification or {}).get("verdict", "not_verified"),
            "blocked_reason": (s.blocked_reason or "")[:70],
            "local_path": s.local_path or "",
            "on_disk": bool(s.local_path) and (PROJECT / s.local_path).exists(),
        }
        for s in SOURCES
    ]
)
print(f"{len(registry)} registry entries in data/SOURCES.yaml")
display(pd.crosstab(registry["status"], registry["access"], margins=True))
display(registry["verdict"].value_counts().rename("entries").to_frame())
print("blocked entries:")
display(registry.loc[registry["status"] == "blocked", ["id", "blocked_reason"]])
acquired = registry.loc[registry["local_path"].ne("")]
print(f"{len(acquired)} entries record a local_path; {int(acquired['on_disk'].sum())} are on disk")

## The transcription pipeline's state

This is a transcription project rather than a download project. The optical character
recognition bundled with the archive.org scans of the annuals is not usable for numbers: the
1985 volume's cover title itself came out garbled and numeric rows arrive with column
separators merged or dropped, so row and column alignment is gone. No curated machine-readable
transcription of the annuals exists on Zenodo, GitHub or Harvard Dataverse. The series has to
be created by people typing from page images.

So the pipeline's state is the project's state: which of the eight targets has a template,
which templates have been filled, and what the validator says about each filled file. The
`table_seen` column is the honesty flag - it is true only where somebody opened the volume and
found the table, which today is target 1 alone.

Templates and filled forms live under `data/transcription/`, which this repository does not
ship: both directories are created by whoever runs `make templates`, and nothing in `data/`
that is not provenance belongs in version control.

In [ ]:
queue = pd.DataFrame(
    [
        {
            "priority": t.priority,
            "target_id": t.target_id,
            "source_id": t.source_id,
            "table_seen": t.confirmed_present,
            "cross_check_source_id": t.cross_check_source_id or "",
            "expected_unit": t.expected_unit,
            "expected_currency_basis": str(t.expected_currency_basis),
            "expected_territorial_basis": str(t.expected_territorial_basis),
        }
        for t in sorted(TARGETS, key=lambda x: (x.priority, x.target_id))
    ]
)
queue["template_on_disk"] = [
    (TEMPLATES / f"{target_id}.csv").is_file() for target_id in queue["target_id"]
]
queue["fields_json_on_disk"] = [
    (TEMPLATES / f"{target_id}.fields.json").is_file() for target_id in queue["target_id"]
]
display(queue)
print(
    f"{len(TARGETS)} targets; tables somebody has actually opened and seen: "
    f"{int(queue['table_seen'].sum())}"
)
print(f"templates on disk: {int(queue['template_on_disk'].sum())} of {len(queue)}")

FILLED_FILES = []
if FILLED.is_dir():
    FILLED_FILES = sorted(FILLED.glob("*.csv")) + sorted(FILLED.glob("*.json"))
if not TEMPLATES.is_dir() or not any(TEMPLATES.glob("*.csv")):
    print()
    print(TEMPLATES_FIRST)
if not FILLED_FILES:
    print()
    print(FILLED_FIRST)
else:
    print(f"filled files on disk: {len(FILLED_FILES)}")

In [ ]:
if not FILLED_FILES:
    print(FILLED_FIRST)
else:
    summaries, code_counts = [], []
    for path in FILLED_FILES:
        report = validate_file(path)
        summaries.append(
            {
                "file": path.name,
                "rows": report.n_rows,
                "errors": len(report.errors),
                "warnings": len(report.warnings),
                "usable": report.ok,
            }
        )
        for name, count in report.codes().items():
            code_counts.append({"file": path.name, "code": name, "issues": count})
        for issue in report.issues[:10]:
            print(f"{path.name}: {issue}")
    display(pd.DataFrame(summaries))
    if code_counts:
        display(
            pd.DataFrame(code_counts).pivot_table(
                index="code", columns="file", values="issues", fill_value=0
            )
        )
    else:
        print("no issues of any kind were raised")
    print(
        "an error means the file must not be used; a warning means a person must look at "
        "the page and then either fix the cell or record why it is right"
    )

## Double transcription, and the gate it controls

Two independent readings of the same printed table are compared on
`(row_label, column_label, period)`, and the comparison reports the share of jointly
transcribed cells read differently and the share of digit positions that differ, by position.
Digits are aligned from the most significant end, because that is the position a first-digit
test reads; where two readings differ in length, every position past the shorter string counts
as compared and disagreeing, since a dropped digit displaces everything after it.

Files are paired below on the printed table they claim: `source_id`, `edition_year`, `page`
and `table_number`. The transcriber and the transcription date are deliberately not part of
that key, because two readings of one page are supposed to differ in exactly those fields.

**There is no built-in threshold and this notebook does not propose one.** No published
standard exists for an acceptable transcription error rate in this setting, and inventing one
would be the kind of borrowed number this programme refuses.
`gosplan.transcribe.compare.digit_tests_permitted` requires the threshold as an argument, so
whoever runs a digit test has to write down what they consider acceptable and defend it beside
the result. Measuring a rate is not judging it.

In [ ]:
IDENTITY_FIELDS = ("source_id", "edition_year", "page", "table_number")

if not FILLED_FILES:
    print(FILLED_FIRST)
else:
    by_table = {}
    for path in FILLED_FILES:
        rows = read_filled(path)
        if not rows:
            print(f"{path.name}: no data rows")
            continue
        key = tuple(str(rows[0].get(field, "")) for field in IDENTITY_FIELDS)
        by_table.setdefault(key, []).append(path)
    pairs = [(key, paths) for key, paths in by_table.items() if len(paths) > 1]
    print(
        f"{len(by_table)} distinct printed tables across {len(FILLED_FILES)} filled files; "
        f"{len(pairs)} of them read more than once"
    )
    if not pairs:
        print("no table has two independent readings, so no disagreement rate exists yet")
        print("known_traps trap 9: until one does, no digit test may be run on any of them")
    for key, paths in pairs:
        print(f"\n--- {dict(zip(IDENTITY_FIELDS, key, strict=True))}")
        for i, path_a in enumerate(paths):
            for path_b in paths[i + 1 :]:
                report = compare_files(path_a, path_b)
                print(report.summary())
                print(f"same printed table and same cells covered: {report.same_table}")
                display(
                    pd.DataFrame(
                        [
                            {
                                "digit_position": p.position,
                                "compared": p.n_compared,
                                "disagreements": p.n_disagreements,
                                "rate": p.rate,
                            }
                            for p in report.by_position
                        ]
                    )
                )

## Units, currency basis and territorial basis

These three columns exist because two events break comparability in ways that cannot be
recovered after the fact, and a figure separated from them is not recoverable later.

**The 1961 currency reform.** The rouble was redenominated ten to one, so a value series
crossing 1961 shows a discontinuity of exactly one order of magnitude, and later volumes
sometimes restate pre-1961 figures in new roubles and sometimes do not. `currency_basis` is
mandatory on every cell and has **no `unstated` member**: a transcriber who cannot tell leaves
the cell blank and says why in `notes`.

**The 1939-40 territorial changes.** The annexations changed the territory that "USSR" refers
to, and the yearbooks often print both bases side by side. `territorial_basis` is a
table-level field; leaving it unstated is a hard error for any table whose periods span 1939
or 1940 and a warning otherwise.

Two more distinctions the schema encodes and this audit therefore reports rather than smooths
over. A **nil mark is not a zero**: "the phenomenon did not occur" and "the value was zero"
are different statements, so `nil_printed` and `no_data_printed` carry no number at all. And
the marker sets themselves are a **default, not a finding** - each volume prints its own key
of conventional signs, and `parse_printed_number` takes the marker sets as arguments so that a
transcriber who finds a different key can pass it.

In [ ]:
print(
    f"{len(FIELD_NAMES)} template columns: {len(TABLE_FIELDS)} table-level, "
    f"{len(FIELD_NAMES) - len(TABLE_FIELDS)} cell-level"
)
print(f"required (never blank): {len(REQUIRED_FIELDS)}")
print("optional:", [f for f in FIELD_NAMES if f not in REQUIRED_FIELDS])
print("currency_basis:", [m.value for m in CurrencyBasis])
print("territorial_basis:", [m.value for m in TerritorialBasis])
print("confidence:", [m.value for m in Confidence])

if not FILLED_FILES:
    print()
    print(FILLED_FIRST)
else:
    frames = []
    for path in FILLED_FILES:
        rows = read_filled(path)
        if rows:
            frames.append(pd.DataFrame(rows).assign(source_file=path.name))
    cells = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if cells.empty:
        print("every filled file is empty")
    else:
        absent = [c for c in FIELD_NAMES if c not in cells.columns]
        print("schema columns absent from the filled files:", absent)
        for column in ("unit", "currency_basis", "territorial_basis", "confidence"):
            if column in cells.columns:
                display(cells[column].value_counts(dropna=False).rename("cells").to_frame())
        if "value" in cells.columns:
            blank = int((cells["value"].astype(str).str.strip() == "").sum())
            print(f"cells carrying no number: {blank:,} of {len(cells):,}")

## The crop production series, which are the anchor's independent evidence

Two of the estimates of the padded quantity are machine-readable and cover the 1978-1983
window. Neither has a loader in this tree: there is no `clean` package here, because the
project's own numbers come from `transcribe`, and these two arrive as publisher-shaped bulk
files. So what follows is a file-level coverage audit driven by the column names recorded in
the `download_plan` field of `data/SOURCES.yaml`, and every cell checks that those columns are
actually present before using them rather than assuming the documented answer.

They measure three different physical quantities, which is a units problem before it is a
forensics problem. FAOSTAT gives **seed cotton, unginned**, in tonnes, for the USSR under area
code 228 in the Europe file, 1961-1991; the Uzbek series begins only in 1992. USDA gives
**cotton lint** in thousands of 480 lb bales, USSR 1960-1986 and Uzbekistan from 1987, with no
"Former Soviet Union" aggregate at all, so union totals for 1987-1991 have to be summed across
successor republics and whichever republics are included is a choice that must be recorded.
The annuals give **raw cotton** in thousand tonnes. Converting between lint and seed cotton
needs a ginning outturn ratio, which is itself a reported quantity.

In [ ]:
FAOSTAT_DIR = RAW / "faostat"
faostat_zips = sorted(FAOSTAT_DIR.glob("*.zip")) if FAOSTAT_DIR.is_dir() else []
if not faostat_zips:
    print(RUN_FIRST if RAW_EMPTY else "no FAOSTAT bulk zips under data/raw/faostat")
for path in faostat_zips:
    with zipfile.ZipFile(path) as archive:
        members = archive.namelist()
        csv_members = [m for m in members if m.lower().endswith(".csv")]
        print(f"{path.name}: {path.stat().st_size:,} bytes, members {members}")
        if not csv_members:
            print("  no CSV member")
            continue
        with archive.open(csv_members[0]) as handle:
            frame = pd.read_csv(handle, encoding="latin-1")
    needed = ["Area Code", "Item Code", "Element Code"]
    absent = [c for c in needed if c not in frame.columns]
    if absent:
        print(f"  header differs from the one data/SOURCES.yaml describes; absent {absent}")
        print(f"  columns as read: {list(frame.columns)[:12]}")
        continue
    year_columns = [
        c
        for c in frame.columns
        if len(str(c)) == 5 and str(c).startswith("Y") and str(c)[1:].isdigit()
    ]
    if year_columns:
        print(
            f"  {len(frame):,} rows, {len(year_columns)} year columns, "
            f"{year_columns[0]} to {year_columns[-1]}"
        )
    else:
        print(f"  {len(frame):,} rows, no year columns")
    area = frame["Area Code"].astype(str).str.strip().str.lstrip("'")
    element = frame["Element Code"].astype(str).str.strip().str.lstrip("'")
    ussr = frame.loc[(area == "228") & (element == "5510")]
    print(f"  area 228 (USSR), element 5510 (production): {len(ussr)} rows")
    rows = []
    for _, record in ussr.iterrows():
        years = [int(str(c)[1:]) for c in year_columns if pd.notna(record[c])]
        rows.append(
            {
                "item_code": record["Item Code"],
                "item": record.get("Item", ""),
                "unit": record.get("Unit", ""),
                "years_present": len(years),
                "first_year": min(years) if years else None,
                "last_year": max(years) if years else None,
            }
        )
    if rows:
        display(pd.DataFrame(rows))

In [ ]:
USDA_DIR = RAW / "usda_psd"
usda_zips = sorted(USDA_DIR.glob("*.zip")) if USDA_DIR.is_dir() else []
if not usda_zips:
    print(RUN_FIRST if RAW_EMPTY else "no USDA PSD zips under data/raw/usda_psd")
for path in usda_zips:
    with zipfile.ZipFile(path) as archive:
        csv_members = [m for m in archive.namelist() if m.lower().endswith(".csv")]
        print(f"{path.name}: {path.stat().st_size:,} bytes, members {archive.namelist()}")
        if not csv_members:
            print("  no CSV member")
            continue
        with archive.open(csv_members[0]) as handle:
            psd = pd.read_csv(handle)
    needed = ["Country_Name", "Market_Year", "Attribute_Description", "Unit_Description"]
    absent = [c for c in needed if c not in psd.columns]
    if absent:
        print(f"  header differs from the one data/SOURCES.yaml describes; absent {absent}")
        print(f"  columns as read: {list(psd.columns)}")
        continue
    print(
        f"  {len(psd):,} rows, {psd['Country_Name'].nunique():,} country names, "
        f"market years {int(psd['Market_Year'].min())} to {int(psd['Market_Year'].max())}"
    )
    names = psd["Country_Name"].astype(str)
    wanted = psd.loc[names.str.startswith("Union of Soviet") | names.eq("Uzbekistan")]
    print(f"  rows for the USSR and Uzbekistan: {len(wanted):,}")
    if len(wanted):
        display(
            wanted.groupby(["Country_Name", "Attribute_Description"]).agg(
                rows=("Market_Year", "size"),
                first_year=("Market_Year", "min"),
                last_year=("Market_Year", "max"),
                units=("Unit_Description", "nunique"),
            )
        )
        print(
            "no 'Former Soviet Union' name exists in this file, so 1987-1991 union totals "
            "must be summed across successor republics"
        )

## The Hokkaido series, which are the reported side rather than a reconstruction

145 CSV files transcribed from the Narkhoz annuals by the Slavic-Eurasian Research Center.
They are not a Western reconstruction: they are an existing independent transcription of the
same official series this project is queuing for transcription, which is why the data
dictionary says to treat them as a second reading to compare against rather than as a
substitute for reading the printed page.

Two caveats travel with them and both are recorded in the registry. **Missing values are coded
`0.0`**, which is the same error the nil-mark rule exists to prevent arriving through a
different door, so the cell below counts zeros separately from numbers and does not treat a
zero as an observation. And **at least one series' unit label disagrees with its magnitudes**:
one file is labelled millions of roubles while its values are in billions. This audit reports
the labels; it does not correct them.

In [ ]:
SESS_DIR = RAW / "hokudai_sess" / "series"
sess_files = sorted(SESS_DIR.glob("*.csv")) if SESS_DIR.is_dir() else []
if not sess_files:
    print(RUN_FIRST if RAW_EMPTY else "no series CSVs under data/raw/hokudai_sess/series")
else:
    rows = []
    for path in sess_files:
        frame = pd.read_csv(path)
        year_columns = [
            c for c in frame.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4
        ]
        values = frame[year_columns].apply(pd.to_numeric, errors="coerce")
        n_zero = int((values == 0.0).sum().sum())
        n_number = int(values.notna().sum().sum())
        years = [int(str(c).strip()) for c in year_columns]
        labels = frame["UNIT"] if "UNIT" in frame.columns else pd.Series(dtype="object")
        rows.append(
            {
                "file": path.name,
                "series_rows": len(frame),
                "year_columns": len(year_columns),
                "first_year": min(years) if years else None,
                "last_year": max(years) if years else None,
                "cells_with_a_number": n_number - n_zero,
                "cells_coded_zero": n_zero,
                "cells_blank": int(values.isna().sum().sum()),
                "units": ";".join(sorted({str(u) for u in labels})),
            }
        )
    sess = pd.DataFrame(rows)
    display(sess)
    print(
        f"{len(sess)} files, {int(sess['series_rows'].sum()):,} series rows, "
        f"{int(sess['cells_with_a_number'].sum()):,} cells with a number, "
        f"{int(sess['cells_coded_zero'].sum()):,} coded 0.0 (which means missing here)"
    )
    display(sess["units"].value_counts().rename("files").to_frame())

## The Western reconstructions, at file level only

Maddison, Harrison and the World Bank's "Soviet Economic Decline" archive are alternative
reconstructions to be reconciled against the official figures and against each other. They are
**not ground truth**: they were built from the same official inputs with different adjustments
and they disagree with each other, and the spread between them is itself evidence about how
much the official series can be trusted.

There is no loader for any of them in this tree, and two file-format traps say why a loader is
not a five-minute job: Harrison's `.xls` files are old BIFF workbooks that `openpyxl` cannot
read, and the World Bank archive holds MicroTSP `.DB` series files and a Lotus `.WK1` sheet.
So the audit below stops at the file level, which is as far as the code in this tree honestly
reaches. Harrison's plan-fraud workbook is the exception worth naming: a case-level index of
prosecuted Soviet reporting fraud for 1943-1962, structurally the Soviet analogue of the SEC
enforcement labels in the `aaer` project, and therefore a positive-unlabelled problem in a
different period rather than a second anchor.

In [ ]:
WESTERN_DIRS = ("harrison", "maddison", "worldbank")
found_any = False
for name in WESTERN_DIRS:
    directory = RAW / name
    files = sorted(p for p in directory.rglob("*") if p.is_file()) if directory.is_dir() else []
    if not files:
        print(f"data/raw/{name}: nothing on disk")
        continue
    found_any = True
    display(
        pd.DataFrame(
            [
                {
                    "directory": name,
                    "file": p.relative_to(directory).as_posix(),
                    "suffix": p.suffix.lower(),
                    "bytes": p.stat().st_size,
                }
                for p in files
            ]
        )
    )
if not found_any:
    print()
    print(RUN_FIRST if RAW_EMPTY else "none of the Western reconstruction files is on disk")

## What this audit could not check

Stated so that nobody mistakes a clean run for a usable series.

1. **Everything the annuals print.** Not one Soviet figure in this project is machine-readable
   yet. The audit above can say that a template exists and that a filled file passes its
   checks; it cannot say that the digits match the page, and only a second independent reading
   can.
2. **The republic-level source.** *Narodnoe khoziaistvo Uzbekskoi SSR* covers the anchor and no
   digitised copy anyone can open has been located. Target 3 is blocked on a person, not on
   code, and no amount of auditing changes that.
3. **The physical correlate for the anchor window.** The river-withdrawal series that was meant
   to be the check the falsifiers did not control begins in 1992, nine years after the padding
   ended. Until one of the two unopened pages of that database yields a pre-1992 series, the
   independent physical check is FAOSTAT against USDA, and both are outside estimates of the
   same reported quantity rather than a driver nobody could touch.
4. **The anchor's own arithmetic.** The best accessible academic source gives a window total
   that does not reconcile with the per-year figure in the same article. Settling which refers
   to what is a prerequisite for any claim about magnitude, and it is a reading task, not a
   computation.
5. **Anything about distortion.** No digit test, no bunching estimate, no dispersion measure
   and no reconciliation residual appears above, and none should be added here. The gate on the
   first of those is a measured double-transcription disagreement rate and a threshold somebody
   has written down and defended.